# AI Engine - RAG Pipeline Flow

This notebook modularizes the RAG (Retrieval-Augmented Generation) pipeline into distinct logical steps as follows:

> **PDF Upload** → **Text Extraction** → **NLP Preprocessing** → **Chunking + Embeddings** → **Vector Database** → **User Query** → **Similarity Retrieval** → **Transformer Model (T5/BART)** → **Generated Answer** → **Chat Interface (MERN App)**

At the end, it binds these steps to a FastAPI server so your MERN app can communicate with it.

### Step 0: Imports, Setup, and Load Models

In [8]:
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np
import fitz 
import os
import nest_asyncio
import uvicorn

nest_asyncio.apply()

print("Loading Models (Embeddings & T5)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print("[OK] Models Loaded!")

# Global State Variables for Vector Store
document_chunks = []
vector_index = None

Loading Models (Embeddings & T5)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6975.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3673.18it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[OK] Models Loaded!


### Step 1 & 2: PDF Upload & Text Extraction
Extract raw string content from the uploaded bytes.

In [9]:
def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    temp_path = "temp_uploaded.pdf"
    with open(temp_path, "wb") as f:
        f.write(pdf_bytes)
        
    doc = fitz.open(temp_path)
    text = "".join([page.get_text("text") + "\n" for page in doc])
    doc.close()
    os.remove(temp_path)
    return text

### Step 3: NLP Preprocessing, Chunking
Clean the text slightly and split it into manageable overlapping chunks.

In [10]:
def preprocess_and_chunk(text: str) -> list:
    # Standard Preprocessing: Strip excess whitespace
    cleaned_text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
    
    # ENHANCEMENT: Increased chunk_size to 700 to prevent context loss.
    # 700 chars roughly matches the 256 token limit of our MiniLM embedding model.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
    chunks = text_splitter.split_text(cleaned_text)
    return chunks

### Step 4 & 5: Embeddings & Vector Database
Convert text chunks into dense vectors and index them into FAISS.

In [11]:
def create_vector_db(chunks: list):
    # Generate embeddings and normalize
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)
    dimension = chunk_embeddings.shape[1]
    
    # Create FAISS Index with Inner Product for normalized Cosine Similarity
    index = faiss.IndexFlatIP(dimension) 
    index.add(np.array(chunk_embeddings))
    return index

### Step 6, 7 & 8: User Query, Similarity Retrieval, & Transformer Model
Take a user's question, find similar text chunks, and ask the T5 model to synthesize an answer.

In [12]:
def answer_user_query(query: str, index, chunks: list) -> str:
    # 1. Similarity Retrieval (ENHANCED)
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    
    # ENHANCEMENT: Retrieve top 4 chunks instead of 3 for more context probability
    distances, indices = index.search(np.array(query_embedding), 4)
    
    retrieved_chunks = [chunks[indices[0][i]] for i in range(4)]
    context = "\n---\n".join(retrieved_chunks)
    
    # 2. Transformer Model Generation (ENHANCED)
    # Using a clearer standard RAG prompt format
    prompt = f"Use the following context to answer the question accurately.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # ENHANCEMENT: Increase max_length to 1024 so context isn't lost.
    # Add repetition_penalty to stop the model from looping text.
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=200, 
        do_sample=False, 
        repetition_penalty=1.2
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer

### Step 9 & 10: Chat Interface (MERN App Connection)
Bind our modularized pipeline to FastAPI endpoints, so the MERN backend can proxy requests.

In [13]:
app = FastAPI()

class QueryRequest(BaseModel):
    question: str

@app.post("/upload/")
async def upload_pdf_endpoint(file: UploadFile = File(...)):
    global document_chunks, vector_index
    
    # 1. & 2. Extractions
    pdf_bytes = await file.read()
    raw_text = extract_text_from_pdf(pdf_bytes)
    
    # 3. NLP Preprocessing & Chunking
    document_chunks = preprocess_and_chunk(raw_text)
    
    # 4. & 5. Vector Database
    vector_index = create_vector_db(document_chunks)
    
    return {"message": "Success"}

@app.post("/ask/")
def ask_question_endpoint(request: QueryRequest):
    # 6., 7., 8., 9.: Query Pipeline
    answer = answer_user_query(request.question, vector_index, document_chunks)
    
    # 10. Returning to MERN Interface
    return {"question": request.question, "answer": answer}

@app.get("/summarize/")
def summarize_document_endpoint():
    if not document_chunks:
        return {"summary": "No document uploaded."}
    
    context = document_chunks[0][:1000] 
    prompt = f"Summarize the following text concisely:\n\n{context}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=150)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"summary": summary}

@app.get("/literature-review/")
def literature_review_endpoint():
    if not document_chunks:
        return {"review": "No document uploaded."}
    
    context = "\n---\n".join(document_chunks[:3]) 
    prompt = f"Provide a comprehensive literature review outlining the main themes and context of the following text:\n\n{context}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, repetition_penalty=1.2)
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"review": review}

@app.get("/key-points/")
def key_points_endpoint():
    if not document_chunks:
        return {"key_points": "No document uploaded."}
    
    context = "\n---\n".join(document_chunks[:3]) 
    prompt = f"Extract the most important key points from the following text as a bulleted list:\n\n{context}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False, repetition_penalty=1.2)
    key_points = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"key_points": key_points}


In [14]:
# Start Server
print("Starting Chat Interface AI Engine on http://127.0.0.1:8000 ...")
from uvicorn import Config, Server
import asyncio

config = Config(app=app, host="127.0.0.1", port=8000)
server = Server(config=config)

# Using await directly bypasses the asyncio.run() conflict in Jupyter
await server.serve()

Starting Chat Interface AI Engine on http://127.0.0.1:8000 ...


INFO:     Started server process [8740]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:7529 - "POST /upload HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:7531 - "POST /upload/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:7531 - "GET /summarize/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:7531 - "GET /literature-review/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:4880 - "GET /key-points/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:9358 - "POST /ask/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:9358 - "POST /upload HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:9360 - "POST /upload/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:9360 - "GET /summarize/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:9360 - "GET /literature-review/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:9360 - "GET /key-points/ HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8740]
